# Gibbs sampling with an unknown noise covariance

Estimate a parameter and an unknown $3 \times 3$ noise covariance by
alternating an ensemble Kalman update for the parameter with a conjugate
draw of the covariance written in plain JAX.

## When to use this

Some models have a block for which an ensemble Kalman update is a good
conditional sampler and another block with a closed-form full conditional.
An unknown noise covariance with an inverse-Wishart prior is a common
example, since its full conditional is again inverse-Wishart
(Gelman et al., 2013). Alternating the two in a Gibbs sweep needs no support from
the package beyond one `kalman.update` call; the other step is ordinary user
code. Component-wise iterative EKI for static models with an unknown
measurement error covariance has the same alternating structure
(Botha et al., 2023).

## Setup

A parameter $u \in \mathbb R^2$ is observed through a linear map
$G \in \mathbb R^{3 \times 2}$ in $n = 40$ replicates with a common unknown
noise covariance $R$:

$$
y_r = G u + e_r, \quad e_r \sim \mathcal N(0, R), \quad r = 1, \dots, n,
\qquad
u \sim \mathcal N(0, C_0), \qquad
R \sim \mathcal{IW}(\nu_0, \Psi_0),
$$

with $\nu_0 = 6$ and $\Psi_0 = 0.8\, I_3$, so that the prior mean of $R$ is
$\Psi_0 / (\nu_0 - 3 - 1) = 0.4\, I_3$. $G$, $C_0$ and the true parameter
$u^\dagger$ come from `toy.linear_gaussian`, and the data are generated with

$$
R^\dagger = \begin{pmatrix} 0.5 & 0.2 & 0 \\ 0.2 & 0.4 & 0.1 \\ 0 & 0.1 & 0.3 \end{pmatrix}.
$$

The two full conditionals are

$$
\begin{aligned}
u \mid R, y_{1:n} &\sim \text{the posterior of } u \text{ given }
  \bar y = G u + \bar e, \ \ \bar e \sim \mathcal N(0, R/n), \\
R \mid u, y_{1:n} &\sim \mathcal{IW}\Big(\nu_0 + n,\ \Psi_0 + \sum_{r=1}^{n}
  (y_r - G u)(y_r - G u)^\top\Big),
\end{aligned}
$$

where $\bar y$ is the mean of the replicates.

In [ ]:
import jax
import jax.numpy as jnp

import enskit
from enskit import kalman, maps, toy
from enskit.linalg import DensePSD

problem = toy.linear_gaussian(parameter_dim=2, data_dim=3)
G = problem.forward
R_true = jnp.array([[0.5, 0.2, 0.0], [0.2, 0.4, 0.1], [0.0, 0.1, 0.3]])
n_rep = 40
key = jax.random.key(0)
key, sub = jax.random.split(key)
ys = G(problem.u_true[None])[0] + jax.random.multivariate_normal(
    sub, jnp.zeros(3), R_true, (n_rep,))
y_bar = ys.mean(axis=0)
nu0, Psi0 = 6.0, 0.4 * 2.0 * jnp.eye(3)

## The covariance step

The draw of $R$ uses the fact that if the rows of $Z \in \mathbb R^{\nu \times 3}$
are independent $\mathcal N(0, \Psi^{-1})$, then $Z^\top Z$ is Wishart with
scale $\Psi^{-1}$ and $(Z^\top Z)^{-1} \sim \mathcal{IW}(\nu, \Psi)$. This is
user code and touches nothing in the package.

In [ ]:
def draw_noise_cov(key, u):
    """R | u, ys ~ IW(nu0 + n_rep, Psi0 + sum_r (y_r - G u)(y_r - G u)^T)."""
    resid = ys - G(u[None])[0]
    nu, Psi = nu0 + n_rep, Psi0 + resid.T @ resid
    L = jnp.linalg.cholesky(jnp.linalg.inv(Psi))
    Z = jax.random.normal(key, (int(nu), 3)) @ L.T
    return jnp.linalg.inv(Z.T @ Z)

## The sweep

Each sweep draws 128 fresh particles from the prior of $u$, pushes them
through $G$, and applies one stochastic (`Matheron`) update toward $\bar y$
with noise covariance $R/n$. One particle, picked at random, is taken as the
draw of $u$, and $R$ is then redrawn given that $u$. A particle of a
stochastic update with a finite ensemble is only an approximate draw from
$u \mid R, y_{1:n}$, so this is an approximate Gibbs sampler. Because the
model is linear and Gaussian, an exact draw is also available by conditioning
the Gaussian prior pushed through $G$ and sampling the result, as in
Example 7.

In [ ]:
R = 0.4 * jnp.eye(3)
draws_R, draws_u = [], []
for sweep in range(400):
    key, k_ens, k_upd, k_pick, k_R = jax.random.split(key, 5)
    ens = problem.prior.sample(k_ens, n_particles=128)
    ens = maps.pushforward(ens, G, inputs="u", output="g")
    post = kalman.update(ens, g=y_bar, noise={"g": DensePSD(R / n_rep)},
                         update_rule=kalman.Matheron(), key=k_upd)
    u = post["u"][jax.random.randint(k_pick, (), 0, post.n_particles)]
    R = draw_noise_cov(k_R, u)
    draws_R.append(R)
    draws_u.append(u)

R_mean = jnp.mean(jnp.stack(draws_R[50:]), axis=0)
print("posterior mean of R:\n", R_mean, "\ntruth:\n", R_true)
print("posterior mean of u:", jnp.mean(jnp.stack(draws_u[50:]), axis=0), "truth", problem.u_true)

posterior mean of R:
 [[0.42908759 0.23487536 0.00187554]
 [0.23487536 0.37901036 0.07962969]
 [0.00187554 0.07962969 0.22679423]] 
truth:
 [[0.5 0.2 0. ]
 [0.2 0.4 0.1]
 [0.  0.1 0.3]]
posterior mean of u: [0.66208684 0.94725662] truth [0.61829299 0.89507437]


## What to notice

After discarding the first 50 of 400 sweeps, the estimated posterior mean of
$R$ has diagonal $(0.429, 0.379, 0.227)$ against the true $(0.5, 0.4, 0.3)$,
and it recovers the positive correlations $0.235$ and $0.080$ where the truth
has $0.2$ and $0.1$. The prior mean $0.4\, I_3$ has no off-diagonal terms, so
these come from the 40 replicates. The posterior mean of $u$ is
$(0.662, 0.947)$, near $u^\dagger = (0.618, 0.895)$.

In [ ]:
# checks
assert jnp.allclose(R_mean, R_true, atol=0.2)

## References

- Botha, I., Adams, M. P., Tran, D. K., Bennett, F. R. & Drovandi, C. (2023). Component-wise iterative ensemble Kalman inversion for static Bayesian models with unknown measurement error covariance. *Inverse Problems*, 39(12), 125014.
- Gelman, A., Carlin, J. B., Stern, H. S., Dunson, D. B., Vehtari, A. & Rubin, D. B. (2013). *Bayesian Data Analysis* (3rd ed.). CRC Press.